In [ ]:
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import ClassVar

import matplotlib.pyplot as plt
# numpy is used in the regression cell below (np.array / np.corrcoef). It was
# missing from this import block, so re-running the notebook top to bottom
# raised "NameError: name 'np' is not defined" at that cell even though the
# stored output showed a result from an earlier session.
import numpy as np
import pandas as pd
import seaborn as sns
import yfinance as yf
from kalshi_python import Market
from sklearn.linear_model import LinearRegression

from client.kalshi_client import AuthedApiInstance

# Equity index compared against the CPI surprise.
#
# This was previously "^NDX" (Nasdaq-100) while every label, variable name, and
# the project README described the series as SPX / S&P 500, so the published
# chart and r^2 did not measure what the surrounding text claimed. The ticker and
# the display label are now defined once, here, and every downstream cell reads
# them from these two names, so the two cannot drift apart again.
#
# Set INDEX_TICKER to "^NDX" and INDEX_LABEL to "NDX" if the Nasdaq-100 is what
# you actually want to study.
INDEX_TICKER = "^GSPC"  # S&P 500 cash index
INDEX_LABEL = "SPX"

# Column name for the index move, derived from INDEX_LABEL so the chart axis and
# the DataFrame key always agree with the ticker actually downloaded.
INDEX_CHANGE_COLUMN = f"{INDEX_LABEL} change"
SURPRISE_COLUMN = "CPI surprise (actual - predicted)"

# set seaborn style
sns.set_theme()
sns.set_style("dark")
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
kalshi_api = AuthedApiInstance()

### Compute Kalshi Forecasts and Historical Surprise

In [ ]:
@dataclass
class CPIRelease:
    """
    Create a single dataclass to store information about a particular CPI release.

    Kalshi markets are individual binaries, of the form:
    "Will CPI growth be >0.5% in February?"

    That means we need to consider the pricing of several individual markets,
    with different strikes, to compute the market consensus for a particular month.
    """

    close_date: datetime
    # None until the release settles: markets that have not expired yet carry an
    # empty expiration_value. Callers must check for None before differencing it
    # against a forecast.
    true_cpi_value: float | None
    markets: list[Market]

    # Offset added to a market's floor strike to get the CPI value that market
    # stands for, in percentage points. 0.1 matches the strike ladder spacing:
    # CPI is reported to one decimal place, so "CPI > 0.4" resolves yes only for
    # 0.5 and above, making 0.5 the lowest value the bucket can take.
    #
    # Declared ClassVar so @dataclass treats these as constants rather than as
    # additional __init__ fields.
    STRIKE_OFFSET: ClassVar[float] = 0.1

    # Maximum yes_ask - yes_bid spread, in cents, for a market to be considered
    # liquid enough to price.
    MAX_SPREAD_CENTS: ClassVar[float] = 10

    def target(self, market: Market) -> float:
        """Returns the representative CPI value, in percent, for ``market``.

        Prefers the structured ``floor_strike`` field and falls back to parsing
        the human-readable subtitle only when it is absent.
        """
        if market.floor_strike is not None:
            return market.floor_strike + self.STRIKE_OFFSET

        # Fallback: parse the display string, e.g. ">0.5%" -> 0.5. This depends on
        # presentation text Kalshi can change at any time, so a parse failure is
        # surfaced with the offending value and the market ticker rather than
        # raising a bare ValueError from float() with no context.
        subtitle = (market.subtitle or "").strip()
        try:
            parsed = float(subtitle.replace(">", "").replace("%", ""))
        except ValueError as exc:
            raise ValueError(
                f"Cannot read a strike from market {market.ticker!r}: "
                f"floor_strike is None and subtitle {subtitle!r} is not numeric."
            ) from exc
        return parsed + self.STRIKE_OFFSET

    def kalshi_consensus(self) -> float:
        """Returns the probability-weighted expected CPI value, in percent.

        Walks the strikes from highest to lowest. Because the markets are
        "CPI > X" binaries, the mid price rises monotonically as the strike
        falls, so ``probability - accounted_probability`` is the incremental
        probability mass sitting in the bucket between this strike and the next
        higher one.
        """
        sorted_markets = sorted(self.markets, key=self.target)

        consensus = 0.0
        # Accumulated in Kalshi cents, i.e. percentage points on a 0-100 scale,
        # because yes_bid / yes_ask are quoted in cents.
        accounted_probability = 0.0

        for market in reversed(sorted_markets):
            # Filter out illiquid markets
            if market.yes_ask - market.yes_bid > self.MAX_SPREAD_CENTS:
                continue
            probability = (market.yes_ask + market.yes_bid) / 2
            unaccounted_probability = max(0, probability - accounted_probability)

            target = self.target(market)
            consensus += target * unaccounted_probability / 100
            accounted_probability += unaccounted_probability

        if accounted_probability <= 0:
            # Every market was filtered out as illiquid, or the whole strike
            # ladder is quoted 0/0. There is no consensus to report. The previous
            # code reached `consensus /= accounted_probability` in exactly this
            # case, because 0 satisfies `< 1`, and raised ZeroDivisionError.
            raise ValueError(
                f"No liquid markets for release closing {self.close_date:%Y-%m-%d}: "
                f"all {len(self.markets)} markets exceeded the "
                f"{self.MAX_SPREAD_CENTS}c spread filter or were unquoted."
            )

        # Renormalise when the ladder does not account for the full 100% of
        # probability, which leaves the result as E[CPI | CPI > lowest strike].
        #
        # Two bugs are fixed here. The threshold was `< 1`, comparing a 0-100
        # percentage-point total against 1, so the branch was dead for every
        # realistic ladder: any total from 1 to 100 skipped normalisation
        # entirely. The divisor was also the raw percentage-point total rather
        # than the fraction, which is off by a factor of 100 -- dividing a
        # consensus of ~0.5 by 95 would have produced ~0.005 instead of ~0.53.
        if accounted_probability < 100:
            consensus /= accounted_probability / 100

        return consensus

In [ ]:
# Get summary statistics
series_ticker = 'CPI'

markets_in_series = kalshi_api.get_markets(
    series_ticker=series_ticker, limit=1000
).markets

cpi_releases_by_ticker = {}

for market in markets_in_series:
    event_ticker = market.event_ticker

    if "CPI-" not in event_ticker:
        continue
    if event_ticker not in cpi_releases_by_ticker:
        close_date = datetime.fromisoformat(market.close_time)

        # expiration_value is an empty string until the release settles, and can
        # also carry a non-numeric placeholder. Anything unparseable becomes
        # None, which the surprise loop below skips, rather than aborting the
        # whole ingest with a ValueError from float().
        raw_expiration = (market.expiration_value or "").replace("%", "").strip()
        try:
            true_cpi_value = float(raw_expiration) if raw_expiration else None
        except ValueError:
            true_cpi_value = None

        cpi_releases_by_ticker[event_ticker] = CPIRelease(
            close_date, true_cpi_value, []
        )

    cpi_releases_by_ticker[event_ticker].markets.append(market)

print(
    f"Loaded {len(cpi_releases_by_ticker)} CPI releases "
    f"from {len(markets_in_series)} markets."
)

### Compare to Index Movements

The index is selected by `INDEX_TICKER` in the first cell. It ships as `^GSPC`
(S&P 500) to match the `SPX` labels used throughout; change both `INDEX_TICKER`
and `INDEX_LABEL` together to study a different index.

In [ ]:
# Hourly bars for the configured index. `INDEX_TICKER` replaces the previously
# hardcoded "^NDX" so the series matches the SPX labelling used downstream.
index_prices = yf.Ticker(INDEX_TICKER).history(
    start="2022-01-01",
    end=datetime.now().strftime("%Y-%m-%d"),
    interval="1h",
)

if index_prices.empty:
    # yfinance returns an empty frame rather than raising when a ticker is
    # unknown, rate-limited, or the interval/window combination is rejected.
    # Failing here names the cause instead of letting the next cell die on an
    # out-of-bounds .iloc[-1].
    raise RuntimeError(
        f"yfinance returned no rows for {INDEX_TICKER}. Check the ticker symbol "
        f"and note that the 1h interval only covers roughly the last 730 days."
    )

index_prices.index = pd.to_datetime(index_prices.index)
index_prices["Delta"] = index_prices["Close"] - index_prices["Open"]

print(f"{INDEX_TICKER}: {len(index_prices)} hourly bars, "
      f"{index_prices.index.min()} to {index_prices.index.max()}")

In [ ]:
kalshi_surprise = []
index_change = []
skipped = []

for ticker, cpi_release in cpi_releases_by_ticker.items():
    is_future = cpi_release.close_date > datetime.now().astimezone(timezone.utc)
    is_recent = cpi_release.close_date >= datetime(2022, 6, 1, tzinfo=timezone.utc)
    if is_future or not is_recent:
        continue

    # A release that has closed but not yet settled has no expiration value, so
    # there is nothing to subtract the consensus from. Previously this reached
    # `None - float(...)` and aborted the whole loop with a TypeError.
    if cpi_release.true_cpi_value is None:
        skipped.append((ticker, "no settled expiration value"))
        continue

    before = index_prices[index_prices.index < cpi_release.close_date]
    after = index_prices[index_prices.index >= cpi_release.close_date]
    if before.empty or after.empty:
        # The release falls outside the downloaded price window. .iloc[-1] /
        # .iloc[0] on an empty frame raises IndexError, which previously killed
        # the loop instead of skipping the one unusable release.
        skipped.append((ticker, "release outside the downloaded price window"))
        continue

    try:
        consensus = cpi_release.kalshi_consensus()
    except ValueError as exc:
        # Raised when every market in the ladder was filtered out as illiquid.
        skipped.append((ticker, str(exc)))
        continue

    last_point = before.iloc[-1]
    next_point = after.iloc[0]

    kalshi_surprise.append(cpi_release.true_cpi_value - consensus)
    index_change.append(next_point["Open"] - last_point["Close"])

surprise_vs_index = pd.DataFrame(
    {SURPRISE_COLUMN: kalshi_surprise, INDEX_CHANGE_COLUMN: index_change}
)

print(f"Usable releases: {len(surprise_vs_index)}")
for ticker, reason in skipped:
    print(f"  skipped {ticker}: {reason}")

In [ ]:
sns.regplot(data=surprise_vs_index, x=SURPRISE_COLUMN, y=INDEX_CHANGE_COLUMN)

In [ ]:
surprise_array = np.asarray(kalshi_surprise, dtype=float).reshape(-1, 1)
change_array = np.asarray(index_change, dtype=float)

surprise_to_index = LinearRegression().fit(surprise_array, change_array)

# Check r^2 correlation - the closer this is to one, the better
# indicator Kalshi surprise is of the index change.
#
# The third positional argument to np.corrcoef is `rowvar`, not a lag or a ddof.
# The previous call passed 1 there, which silently re-specified the default and
# read as if it meant something. It is dropped rather than kept.
r_squared = np.corrcoef(surprise_array.ravel(), change_array)[0, 1] ** 2
print(f"n = {len(change_array)}   r^2 = {r_squared:.4f}")
r_squared

### Index Move Calculator

In [ ]:
def predicted_index_change(cpi_forecast_for_next_release: float) -> float:
    """
    Given a CPI forecast for the next release, predict the index change.

    Returns the predicted point move so the caller can use the number, rather
    than only printing it. The previous version returned None implicitly.
    """
    now = datetime.now().astimezone(timezone.utc)
    future_cpi_releases = [
        r for r in cpi_releases_by_ticker.values() if r.close_date > now
    ]
    if not future_cpi_releases:
        # Indexing [0] on an empty list raised a bare IndexError that gave no
        # hint that the market list simply had no open CPI releases left.
        raise ValueError(
            "No CPI releases with a close date in the future. Re-run the market "
            "fetch cell, or check that the CPI series still has open markets."
        )

    next_cpi_release = min(future_cpi_releases, key=lambda x: x.close_date)

    surprise = cpi_forecast_for_next_release - next_cpi_release.kalshi_consensus()
    # Same 2-D shape the model was fitted with, so scikit-learn does not warn
    # about a feature-count mismatch.
    expected_change = surprise_to_index.predict(np.asarray([[surprise]], dtype=float))

    print(
        f"Your forecast implies market surprise of {surprise:.2f}%. "
        f"Using the regression shown above,\n"
        f"              this would cause the {INDEX_LABEL} to move by "
        f"approximately {expected_change[0]:.0f}."
    )
    return float(expected_change[0])

In [ ]:
# 0.5% is passed as an example, but you can pass whatever
# you think the next CPI release will be
predicted_index_change(0.5)